In [5]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# =============================================================================
# CONFIG (EDIT THIS)
# =============================================================================
BASE_DIR = Path.cwd()

SUBMISSIONS_PATH = str(BASE_DIR / "sampled.ndjson")
LABELS_PATH      = str(BASE_DIR / "labels_3000.ndjson")
THREAD_PATH      = str(BASE_DIR / "thread_metrics.ndjson")

MAX_ROWS_PER_FILE = 0  # 0 = all rows

TOP_N_SUBREDDITS = 30
TOP_N_TOPICS_OVERALL = 30
TOP_N_TOPICS_FOR_PER_SUB = 15

# For "task detail" section - you can pick tasks explicitly
SELECTED_LABEL_TASK: Optional[str] = None   # set to a string to force selection
SELECTED_THREAD_TASK: Optional[str] = None  # set to a string to force selection


In [6]:
def _exists_or_raise(path: str, label: str) -> None:
    p = Path(path).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(f"{label} file not found: {p}")

def _max_rows_opt(x: int) -> Optional[int]:
    return None if int(x) == 0 else int(x)

def stringify_unhashables(df: pd.DataFrame) -> pd.DataFrame:
    def make_hashable(x: Any) -> Any:
        if isinstance(x, (list, dict, set, tuple)):
            try:
                return json.dumps(x, sort_keys=True, ensure_ascii=False)
            except TypeError:
                return str(x)
        return x

    if df.empty:
        return df

    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    for c in obj_cols:
        df[c] = df[c].map(make_hashable)
    return df

def normalize_subreddit(x: Any) -> Optional[str]:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return None

    s = s.replace("\\", "/").strip()
    s = re.sub(r"^/r/", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^r/", "", s, flags=re.IGNORECASE)
    s = s.strip().lower()
    return s or None

def maybe_epoch_to_datetime(s: pd.Series) -> pd.Series:
    if s is None or getattr(s, "empty", False):
        return s
    s2 = pd.to_numeric(s, errors="coerce")
    if s2.notna().sum() == 0:
        return s
    unit = "ms" if float(s2.dropna().median()) > 1e12 else "s"
    return pd.to_datetime(s2, unit=unit, utc=True, errors="coerce")

def first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def infer_abstain_mask(df: pd.DataFrame) -> pd.Series:
    if df is None or df.empty:
        return pd.Series(dtype=bool)

    mask = pd.Series(False, index=df.index)

    abstain_cols = [c for c in df.columns if re.search(r"abstain", str(c), flags=re.IGNORECASE)]
    for c in abstain_cols:
        try:
            s = df[c]
            if s.dtype == bool:
                mask = mask | s.fillna(False)
            else:
                s_num = pd.to_numeric(s, errors="coerce")
                if s_num.notna().mean() > 0.2:
                    mask = mask | (s_num.fillna(0) != 0)
                else:
                    s_str = s.astype("string")
                    mask = mask | (s_str.str.strip().str.lower().isin({"abstain", "true", "yes"}))
        except Exception:
            continue

    for c in ["result.label", "result.score", "result_label", "result_score", "label", "score"]:
        if c in df.columns:
            try:
                s_str = df[c].astype("string").str.strip().str.lower()
                mask = mask | s_str.eq("abstain")
            except Exception:
                continue

    return mask.fillna(False)

def add_label_category_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    out = df.copy()
    out["is_abstain"] = infer_abstain_mask(out)

    raw_label_col = first_existing_col(out, ["result.label", "result.score", "label", "score"])
    if raw_label_col and raw_label_col in out.columns:
        raw = out[raw_label_col].astype("string").str.strip()
        raw_norm = raw.str.lower()
        label_cat = raw_norm.where(~out["is_abstain"], other="abstain")
        label_cat = label_cat.fillna("missing")
    else:
        label_cat = pd.Series("missing", index=out.index)
        label_cat = label_cat.where(~out["is_abstain"], other="abstain")

    out["label_category"] = label_cat.astype("string")
    return out

def read_ndjson_objects(path: str, max_rows: Optional[int] = None) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def load_submissions(path: str, max_rows: Optional[int] = None) -> pd.DataFrame:
    p = str(Path(path).expanduser().resolve())
    if max_rows is None:
        df = pd.read_json(p, lines=True)
    else:
        objs = read_ndjson_objects(p, max_rows=max_rows)
        df = pd.DataFrame(objs)

    df = stringify_unhashables(df)

    c_utc = first_existing_col(df, ["created_utc", "createdUTC", "created"])
    if c_utc:
        df["created_dt"] = maybe_epoch_to_datetime(df[c_utc])

    return df

def load_nested_as_flat(path: str, max_rows: Optional[int] = None, sep: str = ".") -> pd.DataFrame:
    p = str(Path(path).expanduser().resolve())
    objs = read_ndjson_objects(p, max_rows=max_rows)
    if not objs:
        return pd.DataFrame()

    df = pd.json_normalize(objs, sep=sep)
    df = stringify_unhashables(df)

    for col in ["meta.created_utc", "meta.submission_created_utc", "created_utc", "submission_created_utc"]:
        if col in df.columns:
            df[col.replace(".", "_") + "_dt"] = maybe_epoch_to_datetime(df[col])

    if "result.score" in df.columns:
        df["result_score_num"] = safe_numeric(df["result.score"])
    if "result.label" in df.columns:
        df["result_label_num"] = safe_numeric(df["result.label"])
    if "result.confidence" in df.columns:
        df["result_conf_num"] = safe_numeric(df["result.confidence"])

    df = add_label_category_columns(df)
    return df


In [7]:
def compute_subreddit_kpis(
        df_sub: pd.DataFrame,
        df_lab: pd.DataFrame,
        subreddit_col_sub: str,
        subreddit_col_lab: str,
        submission_id_col_sub: Optional[str],
        comment_id_col_lab: Optional[str],
        num_comments_col_sub: Optional[str],
        *,
        use_distinct_comments: bool = True,
) -> pd.DataFrame:
    if df_sub.empty:
        return pd.DataFrame()

    sub = df_sub.copy()
    lab = df_lab.copy() if not df_lab.empty else pd.DataFrame()

    sub["subreddit_key"] = sub[subreddit_col_sub].map(normalize_subreddit)
    sub = sub.dropna(subset=["subreddit_key"])

    if not lab.empty:
        lab["subreddit_key"] = lab[subreddit_col_lab].map(normalize_subreddit)
        lab = lab.dropna(subset=["subreddit_key"])

    g_sub = (
        sub.groupby("subreddit_key", dropna=True)
        .size()
        .rename("submissions_count")
        .reset_index()
    )

    if submission_id_col_sub and submission_id_col_sub in sub.columns:
        g_ids = (
            sub.groupby("subreddit_key", dropna=True)[submission_id_col_sub]
            .nunique(dropna=True)
            .rename("distinct_submission_ids")
            .reset_index()
        )
    else:
        g_ids = pd.DataFrame(columns=["subreddit_key", "distinct_submission_ids"])

    g_nc = pd.DataFrame(columns=["subreddit_key", "num_comments_sum", "num_comments_mean", "num_comments_median"])
    if num_comments_col_sub and num_comments_col_sub in sub.columns:
        tmp = sub[["subreddit_key", num_comments_col_sub]].copy()
        tmp[num_comments_col_sub] = pd.to_numeric(tmp[num_comments_col_sub], errors="coerce")
        g_nc = (
            tmp.groupby("subreddit_key", dropna=True)[num_comments_col_sub]
            .agg(num_comments_sum="sum", num_comments_mean="mean", num_comments_median="median")
            .reset_index()
        )

    g_lab = pd.DataFrame(columns=["subreddit_key", "label_rows", "labeled_comment_ids", "abstain_rows", "abstain_share_of_label_rows"])
    if not lab.empty:
        g_rows = (
            lab.groupby("subreddit_key", dropna=True)
            .size()
            .rename("label_rows")
            .reset_index()
        )

        if "is_abstain" in lab.columns:
            g_ab = (
                lab.groupby("subreddit_key", dropna=True)["is_abstain"]
                .sum()
                .rename("abstain_rows")
                .reset_index()
            )
        else:
            g_ab = pd.DataFrame(columns=["subreddit_key", "abstain_rows"])

        comment_id_ok = (
                use_distinct_comments
                and comment_id_col_lab
                and comment_id_col_lab in lab.columns
                and lab[comment_id_col_lab].notna().mean() > 0.05
        )

        if comment_id_ok:
            cid = lab[comment_id_col_lab]
            cid = cid.where(cid.notna(), np.nan).astype("string")
            g_cids = (
                lab.assign(_cid=cid)
                .dropna(subset=["_cid"])
                .groupby("subreddit_key", dropna=True)["_cid"]
                .nunique()
                .rename("labeled_comment_ids")
                .reset_index()
            )
        else:
            g_cids = pd.DataFrame(columns=["subreddit_key", "labeled_comment_ids"])

        g_lab = g_rows.merge(g_cids, how="left", on="subreddit_key").merge(g_ab, how="left", on="subreddit_key")
        g_lab["abstain_rows"] = g_lab["abstain_rows"].fillna(0).astype(int)
        g_lab["abstain_share_of_label_rows"] = g_lab["abstain_rows"] / g_lab["label_rows"].replace({0: np.nan})

    out = (
        g_sub
        .merge(g_ids, how="left", on="subreddit_key")
        .merge(g_nc, how="left", on="subreddit_key")
        .merge(g_lab, how="left", on="subreddit_key")
    )

    out["label_rows"] = out.get("label_rows", 0).fillna(0).astype(int)
    out["labeled_comment_ids"] = out.get("labeled_comment_ids", np.nan)
    out["comments_count"] = out["labeled_comment_ids"].fillna(0).astype(int)

    out["rows_per_labeled_comment"] = out["label_rows"] / out["comments_count"].replace({0: np.nan})

    if "num_comments_sum" in out.columns:
        out["engagement_from_submissions"] = out["num_comments_sum"] / out["submissions_count"].replace({0: np.nan})
    else:
        out["engagement_from_submissions"] = np.nan

    out["engagement_from_labels"] = out["comments_count"] / out["submissions_count"].replace({0: np.nan})

    out = out.rename(columns={"subreddit_key": "subreddit"})
    out = out.sort_values(["submissions_count", "comments_count"], ascending=False)
    return out


def compute_topics_overall(df_sub: pd.DataFrame, topic_col: str) -> pd.DataFrame:
    tmp = df_sub[[topic_col]].dropna()
    return tmp[topic_col].value_counts().rename_axis("topic").reset_index(name="count")


def compute_topics_by_subreddit(
        df_sub: pd.DataFrame,
        subreddit_col: str,
        topic_col: str,
        top_n_topics: int = 20,
) -> pd.DataFrame:
    tmp = df_sub[[subreddit_col, topic_col]].dropna()
    out = (
        tmp.groupby([subreddit_col, topic_col], dropna=True)
        .size()
        .rename("count")
        .reset_index()
        .rename(columns={subreddit_col: "subreddit", topic_col: "topic"})
    )

    top_topics = out.groupby("topic", dropna=True)["count"].sum().sort_values(ascending=False).head(top_n_topics).index
    out = out[out["topic"].isin(top_topics)]
    return out.sort_values(["subreddit", "count"], ascending=[True, False])


def compute_task_distributions(
        df_any: pd.DataFrame,
        task_col: str,
        value_cols: List[str],
        conf_col: Optional[str],
        label_cat_col: str = "label_category",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if df_any is None or df_any.empty or task_col not in df_any.columns:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    tmp = df_any.copy()

    if label_cat_col not in tmp.columns:
        tmp = add_label_category_columns(tmp)

    tmp[label_cat_col] = tmp[label_cat_col].astype("string").fillna("missing")

    label_dist = (
        tmp.groupby([task_col, label_cat_col], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .rename(columns={task_col: "task", label_cat_col: "label"})
        .sort_values(["task", "count"], ascending=[True, False])
    )

    value_col = first_existing_col(tmp, value_cols)
    value_dist = pd.DataFrame()
    if value_col is not None and value_col in tmp.columns:
        tmp[value_col] = safe_numeric(tmp[value_col])

        tmp_num = tmp[~tmp["is_abstain"]].copy() if "is_abstain" in tmp.columns else tmp.copy()
        nunique = tmp_num[value_col].nunique(dropna=True)

        if nunique > 0:
            if nunique <= 25:
                value_dist = (
                    tmp_num.dropna(subset=[value_col])
                    .groupby([task_col, value_col], dropna=True)
                    .size()
                    .rename("count")
                    .reset_index()
                    .rename(columns={task_col: "task", value_col: "result"})
                )
            else:
                bins = 20
                tmp2 = tmp_num.dropna(subset=[value_col]).copy()
                tmp2["result_bin"] = pd.cut(tmp2[value_col], bins=bins)
                value_dist = (
                    tmp2.groupby([task_col, "result_bin"], dropna=True)
                    .size()
                    .rename("count")
                    .reset_index()
                    .rename(columns={task_col: "task", "result_bin": "result"})
                )

    conf_summary = pd.DataFrame()
    if conf_col and conf_col in tmp.columns:
        tmp[conf_col] = safe_numeric(tmp[conf_col])
        conf_summary = (
            tmp.groupby(task_col, dropna=True)[conf_col]
            .agg(["count", "mean", "median"])
            .reset_index()
            .rename(columns={task_col: "task", "count": "n"})
        )
        q = (
            tmp.groupby(task_col, dropna=True)[conf_col]
            .quantile([0.1, 0.9])
            .unstack()
            .reset_index()
            .rename(columns={0.1: "p10", 0.9: "p90", task_col: "task"})
        )
        conf_summary = conf_summary.merge(q, on="task", how="left").sort_values("n", ascending=False)

    return label_dist, value_dist, conf_summary


In [8]:
# =============================================================================
# Abschnitt 0 — Load & Status
# =============================================================================
_max_opt = _max_rows_opt(MAX_ROWS_PER_FILE)

_exists_or_raise(SUBMISSIONS_PATH, "Submissions")
_exists_or_raise(LABELS_PATH, "Labels")
_exists_or_raise(THREAD_PATH, "Thread metrics")

df_sub = load_submissions(SUBMISSIONS_PATH, max_rows=_max_opt)
df_lab = load_nested_as_flat(LABELS_PATH, max_rows=_max_opt)
df_thr = load_nested_as_flat(THREAD_PATH, max_rows=_max_opt)

print("cwd:", Path.cwd())
print("Submissions:", Path(SUBMISSIONS_PATH).expanduser().resolve())
print("Labels:", Path(LABELS_PATH).expanduser().resolve())
print("Thread metrics:", Path(THREAD_PATH).expanduser().resolve())

print(f"Submissions rows:     {len(df_sub):,}")
print(f"Labels rows:          {len(df_lab):,}")
print(f"Thread metrics rows:  {len(df_thr):,}")

display(df_sub.head(3))
display(df_lab.head(3))
display(df_thr.head(3))


cwd: /Users/arthur/DataspellProjects/reddit-l
Submissions: /Users/arthur/DataspellProjects/reddit-l/sampled.ndjson
Labels: /Users/arthur/DataspellProjects/reddit-l/labels_3000.ndjson
Thread metrics: /Users/arthur/DataspellProjects/reddit-l/thread_metrics.ndjson
Submissions rows:     3,125
Labels rows:          4,041
Thread metrics rows:  1,156


,id,author,created_utc,ups,downs,likes,num_comments,selftext,title,subreddit,best_topic_index,best_topic,best_topic_similarity,created_dt
0,1dc82va,clonedhuman,1717977620000,1,0,NaN,2,,Republicans Try To Block 4 Million Workers Fro...,politics,2,The federal minimum wage should be increased.,0.404026,2024-06-10 00:00:20+00:00
1,1dcetz6,Kate_Matthews,1718000389000,1071,0,NaN,96,,US pushes for $50 billion loan to Ukraine usin...,politics,3,The US should provide financial and military a...,0.616209,2024-06-10 06:19:49+00:00
2,1dch1vs,LbextDipsBenchPlatty,1718010068000,1,0,NaN,1,,The risks of AI could be catastrophic. We shou...,politics,17,Artificial Intelligence should replace humans ...,0.517387,2024-06-10 09:01:08+00:00


,comment_id,body,parent_id,comment_index,task,arguments,arguments_confidence,arguments_error,result.task,result.score,result.confidence,meta.thread_id,meta.submission_id,meta.root_id,meta.parent_id,meta.link_id,meta.user,meta.score,meta.created_utc,meta.permalink,meta.subreddit,meta.depth,meta.submission_title,meta.submission_selftext,meta.submission_author,meta.submission_score,meta.submission_created_utc,meta.submission_subreddit,meta.submission_url,meta.best_topic_index,meta.best_topic_similarity,result.label,result.error,meta_created_utc_dt,meta_submission_created_utc_dt,result_score_num,result_label_num,result_conf_num,is_abstain,label_category
0,l7w2bxm,"\nAs a reminder, this subreddit [is for civil ...",1dc82va,0,stance_intensity,"[{""claim"": ""The subreddit is for civil discuss...",1.0,None,stance_intensity,6,0.95,1dc82va,1dc82va,l7w2bxm,1dc82va,1dc82va,AutoModerator,1,1717977620,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,NaN,NaN,2024-06-10 00:00:20+00:00,2024-06-10 00:00:20+00:00,6.0,NaN,0.95,False,missing
1,l7w2bxm,"\nAs a reminder, this subreddit [is for civil ...",1dc82va,0,civility,"[{""claim"": ""The subreddit is for civil discuss...",1.0,None,civility,NaN,1.00,1dc82va,1dc82va,l7w2bxm,1dc82va,1dc82va,AutoModerator,1,1717977620,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,6,NaN,2024-06-10 00:00:20+00:00,2024-06-10 00:00:20+00:00,NaN,6.0,1.00,False,6
2,l7w2bxm,"\nAs a reminder, this subreddit [is for civil ...",1dc82va,0,epistemic_modality,"[{""claim"": ""The subreddit is for civil discuss...",1.0,None,epistemic_modality,0.2,0.80,1dc82va,1dc82va,l7w2bxm,1dc82va,1dc82va,AutoModerator,1,1717977620,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,NaN,NaN,2024-06-10 00:00:20+00:00,2024-06-10 00:00:20+00:00,0.2,NaN,0.80,False,missing


,comment_id,body,parent_id,comment_index,arguments,arguments_confidence,arguments_error,source_task,task,current_arguments,history_arguments,comment_id_canon,parent_id_canon,link_id_canon,meta.thread_id,meta.submission_id,meta.root_id,meta.parent_id,meta.link_id,meta.user,meta.score,meta.created_utc,meta.permalink,meta.subreddit,meta.depth,meta.submission_title,meta.submission_selftext,meta.submission_author,meta.submission_score,meta.submission_created_utc,meta.submission_subreddit,meta.submission_url,meta.best_topic_index,meta.best_topic_similarity,source_result.task,source_result.score,source_result.confidence,result.task,result.score,result.confidence,meta_created_utc_dt,meta_submission_created_utc_dt,result_score_num,result_conf_num,is_abstain,label_category
0,l7w2bxm,"\nAs a reminder, this subreddit [is for civil ...",1dc82va,0,"[{""claim"": ""The subreddit is for civil discuss...",1.0,None,stance_intensity,argument_novelty,"[{""claim"": ""The subreddit is for civil discuss...",[],l7w2bxm,1dc82va,1dc82va,1dc82va,1dc82va,l7w2bxm,1dc82va,1dc82va,AutoModerator,1,1717977620,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,stance_intensity,6,0.95,argument_novelty,1.0,1.0,2024-06-10 00:00:20+00:00,2024-06-10 00:00:20+00:00,1.0,1.0,False,1.0
1,l7w2bxm,"\nAs a reminder, this subreddit [is for civil ...",1dc82va,0,"[{""claim"": ""The subreddit is for civil discuss...",1.0,None,stance_intensity,semantic_entropy,"[{""claim"": ""The subreddit is for civil discuss...",[],l7w2bxm,1dc82va,1dc82va,1dc82va,1dc82va,l7w2bxm,1dc82va,1dc82va,AutoModerator,1,1717977620,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,stance_intensity,6,0.95,semantic_entropy,0.6,0.8,2024-06-10 00:00:20+00:00,2024-06-10 00:00:20+00:00,0.6,0.8,False,0.6
2,l7w2fvc,Hi `clonedhuman`. Thank you for participating ...,1dc82va,1,"[{""claim"": ""This article has been submitted to...",1.0,None,stance_intensity,argument_novelty,"[{""claim"": ""This article has been submitted to...",[],l7w2fvc,1dc82va,1dc82va,1dc82va,1dc82va,l7w2fvc,1dc82va,1dc82va,PoliticsModeratorBot,1,1717977666,/r/politics/comments/1dc82va/republicans_try_t...,politics,0,Republicans Try To Block 4 Million Workers Fro...,,clonedhuman,None,1717977620000,politics,None,2,0.404026,stance_intensity,1,0.95,argument_novelty,1.0,1.0,2024-06-10 00:01:06+00:00,2024-06-10 00:00:20+00:00,1.0,1.0,False,1.0


In [9]:
# =============================================================================
# Abschnitt 1 — Detect key columns (wie Dashboard)
# =============================================================================
subreddit_sub = first_existing_col(df_sub, ["subreddit", "meta.subreddit", "subreddit_name_prefixed"])
subreddit_lab = first_existing_col(df_lab, ["meta.subreddit", "subreddit", "meta.subverse", "meta.subverse_name"])
subreddit_thr = first_existing_col(df_thr, ["meta.subreddit", "subreddit", "meta.subverse", "meta.subverse_name"])

topic_sub = first_existing_col(df_sub, ["best_topic", "meta.best_topic", "topic", "meta.topic"])

submission_id_sub = first_existing_col(df_sub, ["id", "submission_id", "name", "link_id"])
comment_id_lab = first_existing_col(df_lab, ["comment_id", "id", "name", "meta.comment_id"])
comment_id_thr = first_existing_col(df_thr, ["comment_id", "id", "name", "meta.comment_id"])

num_comments_sub = first_existing_col(df_sub, ["num_comments", "meta.num_comments", "comment_count"])

task_lab = first_existing_col(df_lab, ["task"])
task_thr = first_existing_col(df_thr, ["task", "source_task"])

conf_lab = first_existing_col(df_lab, ["result_conf_num", "result.confidence"])
conf_thr = first_existing_col(df_thr, ["result_conf_num", "result.confidence"])

value_candidates = ["result_score_num", "result_label_num", "result.score", "result.label"]

print("subreddit_sub:", subreddit_sub)
print("subreddit_lab:", subreddit_lab)
print("subreddit_thr:", subreddit_thr)
print("topic_sub:", topic_sub)
print("submission_id_sub:", submission_id_sub)
print("comment_id_lab:", comment_id_lab)
print("comment_id_thr:", comment_id_thr)
print("num_comments_sub:", num_comments_sub)
print("task_lab:", task_lab)
print("task_thr:", task_thr)
print("conf_lab:", conf_lab)
print("conf_thr:", conf_thr)


subreddit_sub: subreddit
subreddit_lab: meta.subreddit
subreddit_thr: meta.subreddit
topic_sub: best_topic
submission_id_sub: id
comment_id_lab: comment_id
comment_id_thr: comment_id
num_comments_sub: num_comments
task_lab: task
task_thr: task
conf_lab: result_conf_num
conf_thr: result_conf_num


In [10]:
# =============================================================================
# Abschnitt 2 — Subreddit KPIs (wie Tab 1)
# =============================================================================

if df_sub.empty or df_lab.empty:
    print("WARNING: Need both submissions and labels.")
elif not subreddit_sub or not subreddit_lab:
    print("WARNING: Could not detect subreddit columns.")
else:
    kpi = compute_subreddit_kpis(
        df_sub=df_sub,
        df_lab=df_lab,
        subreddit_col_sub=subreddit_sub,
        subreddit_col_lab=subreddit_lab,
        submission_id_col_sub=submission_id_sub,
        comment_id_col_lab=comment_id_lab,
        num_comments_col_sub=num_comments_sub,
        use_distinct_comments=True,
    )

    kpi_show = kpi.head(TOP_N_SUBREDDITS)
    display(kpi_show)

    # Charts
    fig1 = px.bar(kpi_show, x="subreddit", y="submissions_count", title="Submissions per subreddit")
    fig2 = px.bar(kpi_show, x="subreddit", y="comments_count", title="Comments per subreddit (proxy from labels)")
    fig3 = px.bar(kpi_show, x="subreddit", y="engagement_from_submissions", title="Engagement (sum(num_comments)/#submissions)")
    fig4 = px.bar(kpi_show, x="subreddit", y="engagement_from_labels", title="Engagement (distinct labeled comments/#submissions)")

    fig1.show()
    fig2.show()
    fig3.show()
    fig4.show()

    # Abstain charts
    if "abstain_rows" in kpi_show.columns:
        fig5 = px.bar(kpi_show, x="subreddit", y="abstain_rows", title="Abstain rows (labels)")
        fig6 = px.bar(kpi_show, x="subreddit", y="abstain_share_of_label_rows", title="Abstain share of label rows")
        fig5.show()
        fig6.show()

    # Diagnostic coverage
    tmp = df_lab.copy()
    tmp["subreddit_norm"] = tmp[subreddit_lab].map(normalize_subreddit)
    cov = (
        tmp["subreddit_norm"]
        .value_counts(dropna=True)
        .head(50)
        .rename_axis("subreddit")
        .reset_index(name="label_rows")
    )
    display(cov)
    px.bar(cov.head(20), x="subreddit", y="label_rows", title="Label rows coverage (top 20)").show()


,subreddit,submissions_count,distinct_submission_ids,num_comments_sum,num_comments_mean,num_comments_median,label_rows,labeled_comment_ids,abstain_rows,abstain_share_of_label_rows,comments_count,rows_per_labeled_comment,engagement_from_submissions,engagement_from_labels
1,askreddit,2019,2019,29652,14.686478,5.0,0,NaN,NaN,NaN,0,NaN,14.686478,0.000000
11,worldnews,436,436,62225,142.717890,6.0,0,NaN,NaN,NaN,0,NaN,142.717890,0.000000
3,conservative,239,239,5907,24.715481,1.0,2998,429.0,477.0,0.159106,429,6.988345,24.715481,1.794979
2,changemyview,154,154,12699,82.461039,1.0,0,NaN,NaN,NaN,0,NaN,82.461039,0.000000
7,politicaldiscussion,111,111,1865,16.801802,1.0,0,NaN,NaN,NaN,0,NaN,16.801802,0.000000
0,amitheasshole,105,105,3079,29.323810,4.0,0,NaN,NaN,NaN,0,NaN,29.323810,0.000000
6,moderatepolitics,19,19,2596,136.631579,35.0,0,NaN,NaN,NaN,0,NaN,136.631579,0.000000
8,politics,16,16,1005,62.812500,15.0,14,2.0,2.0,0.142857,2,7.000000,62.812500,0.125000
5,liberal,14,14,302,21.571429,0.0,0,NaN,NaN,NaN,0,NaN,21.571429,0.000000
4,debate,9,9,22,2.444444,1.0,0,NaN,NaN,NaN,0,NaN,2.444444,0.000000


,subreddit,label_rows
0,conservative,2998
1,truereddit,1015
2,politics,14
3,socialjustice,14


In [11]:
# =============================================================================
# Abschnitt 3 — Topics (wie Tab 2)
# =============================================================================

if df_sub.empty:
    print("WARNING: submissions empty.")
elif not subreddit_sub or not topic_sub:
    print("WARNING: could not detect subreddit/topic columns.")
else:
    overall = compute_topics_overall(df_sub, topic_col=topic_sub)
    display(overall.head(50))
    px.bar(overall.head(TOP_N_TOPICS_OVERALL), x="topic", y="count", title="Overall topics (top)").show()

    by_sub = compute_topics_by_subreddit(
        df_sub=df_sub,
        subreddit_col=subreddit_sub,
        topic_col=topic_sub,
        top_n_topics=TOP_N_TOPICS_FOR_PER_SUB,
    )
    display(by_sub.head(200))

    # Optional: facet plot (can get wide)
    view = by_sub.copy()
    px.bar(view, x="topic", y="count", facet_col="subreddit", facet_col_wrap=2, title="Topics per subreddit (facet)").show()


,topic,count
0,Artificial Intelligence should replace humans ...,472
1,The US should provide financial and military a...,458
2,Social media is a threat to democracy.,374
3,A universal basic income would kill the economy.,284
4,Climate change is one of the greatest threats ...,271
5,Gender-neutral language and stating pronouns a...,200
6,We need stricter gun control laws.,157
7,Prostitution should be illegal.,144
8,Fur clothing should be banned.,127
9,Immigrants should adopt the local language and...,124


,subreddit,topic,count
5,AmItheAsshole,Gender-neutral language and stating pronouns a...,29
4,AmItheAsshole,Fur clothing should be banned.,19
10,AmItheAsshole,The government should not forgive student loan...,11
9,AmItheAsshole,Social media is a threat to democracy.,6
6,AmItheAsshole,Immigrants should adopt the local language and...,5
8,AmItheAsshole,Prostitution should be illegal.,4
11,AmItheAsshole,There should only be vegetarian food in cantines.,4
0,AmItheAsshole,A universal basic income would kill the economy.,2
2,AmItheAsshole,Artificial Intelligence should replace humans ...,2
7,AmItheAsshole,Police officers should wear body cameras.,1


In [12]:
# =============================================================================
# Abschnitt 4 — Task KPIs — Labels (wie Tab 3)
# =============================================================================

if df_lab.empty:
    print("WARNING: labels empty.")
elif not task_lab:
    print("WARNING: no task column detected.")
else:
    tasks = sorted(df_lab[task_lab].dropna().unique().tolist())
    print(f"Tasks found: {len(tasks)}")
    print(tasks)

    # choose task
    selected = SELECTED_LABEL_TASK if SELECTED_LABEL_TASK in tasks else (tasks[0] if tasks else None)
    print("Selected task:", selected)

    dft = df_lab[df_lab[task_lab] == selected].copy()
    if "is_abstain" not in dft.columns or "label_category" not in dft.columns:
        dft = add_label_category_columns(dft)

    # KPI tiles (as prints)
    print(f"Rows: {len(dft):,}")
    print(f"Abstain rows: {int(dft['is_abstain'].sum()):,}")
    print(f"Abstain share: {float(dft['is_abstain'].mean()):.3f}")
    if conf_lab and conf_lab in dft.columns:
        print(f"Mean confidence: {safe_numeric(dft[conf_lab]).mean():.3f}")
    else:
        print("Mean confidence: N/A")

    label_dist, value_dist, conf = compute_task_distributions(
        df_any=dft,
        task_col=task_lab,
        value_cols=value_candidates,
        conf_col=conf_lab,
        label_cat_col="label_category",
    )

    display(label_dist)
    px.bar(label_dist, x="label", y="count", title=f"Label categories — {selected}").show()

    if value_dist.empty:
        print("No usable numeric result field for this task.")
    else:
        display(value_dist.head(200))
        px.bar(value_dist, x="result", y="count", title=f"Numeric distribution — {selected}").show()

    if conf.empty:
        print("No confidence field.")
    else:
        display(conf)

    if conf_lab and conf_lab in dft.columns:
        px.histogram(dft, x=conf_lab, nbins=30, title=f"Confidence histogram — {selected}").show()


Tasks found: 7
['agreement', 'civility', 'epistemic_modality', 'justification_density', 'responsiveness', 'sarcasm', 'stance_intensity']
Selected task: agreement
Rows: 577
Abstain rows: 3
Abstain share: 0.005
Mean confidence: 0.949


,task,label,count
1,agreement,missing,574
0,agreement,abstain,3


,task,result,count
0,agreement,-1.0,121
1,agreement,-0.5,6
2,agreement,0.0,398
3,agreement,0.5,4
4,agreement,1.0,45


,task,n,mean,median,p10,p90
0,agreement,577,0.9487,0.95,0.95,0.95


In [13]:
# =============================================================================
# Abschnitt 5 — Task KPIs — Thread metrics (wie Tab 4)
# =============================================================================

if df_thr.empty:
    print("WARNING: thread metrics empty.")
elif not task_thr:
    print("WARNING: no task/source_task column detected.")
else:
    tasks = sorted(df_thr[task_thr].dropna().unique().tolist())
    print(f"Tasks found: {len(tasks)}")
    print(tasks)

    selected = SELECTED_THREAD_TASK if SELECTED_THREAD_TASK in tasks else (tasks[0] if tasks else None)
    print("Selected task:", selected)

    dft = df_thr[df_thr[task_thr] == selected].copy()
    if "is_abstain" not in dft.columns or "label_category" not in dft.columns:
        dft = add_label_category_columns(dft)

    print(f"Rows: {len(dft):,}")
    print(f"Abstain rows: {int(dft['is_abstain'].sum()):,}")
    print(f"Abstain share: {float(dft['is_abstain'].mean()):.3f}")
    if conf_thr and conf_thr in dft.columns:
        print(f"Mean confidence: {safe_numeric(dft[conf_thr]).mean():.3f}")
    else:
        print("Mean confidence: N/A")

    label_dist, value_dist, conf = compute_task_distributions(
        df_any=dft,
        task_col=task_thr,
        value_cols=value_candidates,
        conf_col=conf_thr,
        label_cat_col="label_category",
    )

    display(label_dist)
    px.bar(label_dist, x="label", y="count", title=f"Label categories — {selected}").show()

    if value_dist.empty:
        print("No usable numeric metric field for this task.")
    else:
        display(value_dist.head(200))
        px.bar(value_dist, x="result", y="count", title=f"Numeric distribution — {selected}").show()

    if conf.empty:
        print("No confidence field.")
    else:
        display(conf)

    if conf_thr and conf_thr in dft.columns:
        px.histogram(dft, x=conf_thr, nbins=30, title=f"Confidence histogram — {selected}").show()


Tasks found: 2
['argument_novelty', 'semantic_entropy']
Selected task: argument_novelty
Rows: 578
Abstain rows: 0
Abstain share: 0.000
Mean confidence: 0.918


,task,label,count
1,argument_novelty,0.5,234
4,argument_novelty,1.0,225
0,argument_novelty,0.0,116
3,argument_novelty,0.8,2
2,argument_novelty,0.6,1


,task,result,count
0,argument_novelty,0.0,116
1,argument_novelty,0.5,234
2,argument_novelty,0.6,1
3,argument_novelty,0.8,2
4,argument_novelty,1.0,225


,task,n,mean,median,p10,p90
0,argument_novelty,578,0.918166,1.0,0.8,1.0


In [14]:
# =============================================================================
# Abschnitt 6 — Boxplots per subreddit (wie Tab 5)
# =============================================================================

SOURCE = "Labels"  # "Labels" or "Thread metrics"
N_SUBREDDITS = 10  # top N subreddits by rows
FILTER_TASKS: List[str] = []  # empty = all tasks
Y_MODE = "Confidence"  # "Confidence" or "Numeric"

df_src = df_lab if SOURCE == "Labels" else df_thr
subreddit_src = subreddit_lab if SOURCE == "Labels" else subreddit_thr
task_src = task_lab if SOURCE == "Labels" else task_thr
conf_src = conf_lab if SOURCE == "Labels" else conf_thr

if df_src.empty:
    print("Selected dataset is empty.")
elif not subreddit_src:
    print("No subreddit column detected in selected dataset.")
else:
    dfp = df_src.copy()
    dfp["subreddit_norm"] = dfp[subreddit_src].map(normalize_subreddit)

    if "label_category" not in dfp.columns or "is_abstain" not in dfp.columns:
        dfp = add_label_category_columns(dfp)

    # top subreddits
    top_subs = dfp["subreddit_norm"].value_counts(dropna=True).head(N_SUBREDDITS).index.tolist()
    dfp = dfp[dfp["subreddit_norm"].isin(top_subs)].copy()

    # optional task filter
    if FILTER_TASKS and task_src and task_src in dfp.columns:
        dfp = dfp[dfp[task_src].isin(FILTER_TASKS)].copy()

    if Y_MODE == "Confidence":
        if not conf_src or conf_src not in dfp.columns:
            print("No confidence column available.")
        else:
            dfp["_y"] = safe_numeric(dfp[conf_src])
            fig = px.box(
                dfp.dropna(subset=["_y"]),
                x="subreddit_norm",
                y="_y",
                color="label_category",
                points="outliers",
                title=f"{SOURCE}: Confidence boxplots (top {N_SUBREDDITS} subs)",
            )
            fig.show()

    else:
        val_col = first_existing_col(dfp, value_candidates)
        if not val_col or val_col not in dfp.columns:
            print("No usable numeric result column.")
        else:
            dfp["_y"] = safe_numeric(dfp[val_col])

            abstain_rate = float(dfp["is_abstain"].mean()) if "is_abstain" in dfp.columns and len(dfp) else np.nan
            print(f"Abstain share in slice (excluded from numeric box): {abstain_rate:.3f}")

            df_num = dfp[(~dfp["is_abstain"]) & (dfp["_y"].notna())].copy()
            fig = px.box(
                df_num,
                x="subreddit_norm",
                y="_y",
                points="outliers",
                title=f"{SOURCE}: Numeric boxplots (top {N_SUBREDDITS} subs)",
            )
            fig.show()

    # Abstain overview for the slice
    ab = (
        dfp.groupby(["subreddit_norm", "label_category"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["subreddit_norm", "count"], ascending=[True, False])
    )
    display(ab)
    px.bar(ab, x="subreddit_norm", y="count", color="label_category", title=f"{SOURCE}: Abstain/label categories in slice").show()


,subreddit_norm,label_category,count
8,conservative,missing,1699
7,conservative,abstain,477
1,conservative,1,244
2,conservative,2,188
0,conservative,0,180
5,conservative,5,93
3,conservative,3,82
6,conservative,6,28
4,conservative,4,7
12,politics,missing,8
